# Step 5E.1 — CTI/native overlap audit (Phase 1)

Дослідницьке питання: чому приріст від CTI-контексту сильно відрізняється між
CIC-IDS2017 (ΔAP(F1-F0) = +0.263659) та UNSW-NB15 (ΔAP(F1-F0) = +0.010317)?

Це Phase 1 з трифазного дизайну (зафіксованого перед запуском): overlap audit -- НЕ
припускаємо наперед, що причина саме overlap між native і CTI ознаками, а перевіряємо це
як одну з гіпотез, вимірюючи, скільки НОВОЇ інформації CTI-контекст фізично може внести
понад те, що вже дає native ознака. Phase 2 (feature attribution) і Phase 3 (targeted
ablation) виконуються тільки після цього і лише якщо Phase 1 дає для них підстави.

Аналіз виконується на TRAIN-спліті (frozen temporal межі, ті самі, що й у Step 5D.2) --
не на test, щоб не змішувати дослідницький аудит з оціночним протоколом.

Порівняння:
- UNSW: native {state, proto, service} vs CTI cve_ctx_service
- CIC: native {protocol} vs CTI cve_ctx_service (CIC не має native service-подібного поля)

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
AUDIT_OUT_DIR = RESULTS_DIR / "step5e1_cti_native_overlap_audit"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True


## 1. Завантаження і frozen train-спліт (ті самі temporal межі, що й Step 5D.2)

In [2]:
FROZEN_BOUNDARIES = {
    "CIC": dict(
        file="cic_ids2017_with_cve_context.parquet", time_col="Timestamp",
        label_col="attack_label", benign_value="BENIGN",
        boundary_1=pd.Timestamp("2017-07-06 09:28:00"),
        boundary_2=pd.Timestamp("2017-07-07 03:23:00"),
    ),
    "UNSW": dict(
        file="unsw_nb15_with_cve_context_dedup.parquet", time_col="Stime",
        label_col="attack_cat", benign_value="Normal",
        boundary_1=pd.Timestamp("2015-02-18 05:36:51"),
        boundary_2=pd.Timestamp("2015-02-18 08:58:36"),
    ),
}

NATIVE_COLS = {"CIC": ["protocol"], "UNSW": ["state", "proto", "service"]}
CTI_COL = "cve_ctx_service"

train_data = {}
for dataset_key, cfg in FROZEN_BOUNDARIES.items():
    df = pd.read_parquet(ENRICHED_DIR / cfg["file"])
    df["_is_attack"] = (df[cfg["label_col"]] != cfg["benign_value"]).astype(int)

    t = pd.to_datetime(df[cfg["time_col"]], unit="s") if dataset_key == "UNSW" and pd.api.types.is_integer_dtype(df[cfg["time_col"]]) \
        else pd.to_datetime(df[cfg["time_col"]])
    train_mask = t < cfg["boundary_1"]
    train_df = df[train_mask].copy()
    train_data[dataset_key] = train_df
    b1 = cfg["boundary_1"]
    print(f"{dataset_key}: train n={len(train_df):,} (frozen boundary_1={b1})")


CIC: train n=1,979,346 (frozen boundary_1=2017-07-06 09:28:00)
UNSW: train n=1,441,568 (frozen boundary_1=2015-02-18 05:36:51)


## 2. Допоміжні функції: entropy (bits), conditional entropy, mutual information, dominant mapping

Categorical association на двох категоріальних ознаках -- НЕ звичайна кореляція. Entropy
рахується у БІТАХ (log2) для інтерпретованості: H(X) -- невизначеність X; H(Y|X) --
скільки невизначеності Y залишається, коли X вже відомий; I(X;Y) = H(Y) - H(Y|X) --
скільки інформації X несе про Y (симетрична величина).

In [3]:
def entropy_bits(series):
    """Shannon entropy H(X) у бітах. NaN трактується як окрема категорія (не відкидається)."""
    s = series.astype("object").where(series.notna(), "__NA__")
    p = s.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def joint_entropy_bits(s1, s2):
    """Спільна ентропія H(X,Y)."""
    a = s1.astype("object").where(s1.notna(), "__NA__")
    b = s2.astype("object").where(s2.notna(), "__NA__")
    joint = pd.Series(list(zip(a, b)))
    p = joint.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def conditional_entropy_bits(target, given):
    """H(target | given) = H(target, given) - H(given)."""
    return joint_entropy_bits(target, given) - entropy_bits(given)

def mutual_information_bits(s1, s2):
    """I(X;Y) = H(X) + H(Y) - H(X,Y) -- симетрична, у бітах."""
    return entropy_bits(s1) + entropy_bits(s2) - joint_entropy_bits(s1, s2)

def normalized_mutual_information(s1, s2):
    """NMI = I(X;Y) / min(H(X), H(Y)) -- 0..1, 1 означає повну (де)термінованість в один бік."""
    h1, h2 = entropy_bits(s1), entropy_bits(s2)
    denom = min(h1, h2)
    if denom == 0:
        return np.nan
    return mutual_information_bits(s1, s2) / denom

def dominant_mapping(df, from_col, to_col):
    """Для кожного значення from_col: домінантне значення to_col, його частка і кількість."""
    rows = []
    for val, group in df.groupby(from_col, observed=True, dropna=False):
        vc = group[to_col].value_counts(normalize=False, dropna=False)
        if len(vc) == 0:
            continue
        dominant_val = vc.index[0]
        count = int(vc.iloc[0])
        total = len(group)
        rows.append(dict(**{from_col: val}, dominant=dominant_val, share=count / total,
                          count=count, group_total=total, n_unique_to=int(group[to_col].nunique(dropna=False))))
    return pd.DataFrame(rows).sort_values("group_total", ascending=False)

print("Допоміжні функції визначено: entropy_bits, joint_entropy_bits, conditional_entropy_bits, "
      "mutual_information_bits, normalized_mutual_information, dominant_mapping")


Допоміжні функції визначено: entropy_bits, joint_entropy_bits, conditional_entropy_bits, mutual_information_bits, normalized_mutual_information, dominant_mapping


## 3. UNSW: {state, proto, service} vs cve_ctx_service

In [4]:
unsw_train = train_data["UNSW"]
unsw_cti_na_rate = unsw_train["cve_ctx_service"].isna().mean()
unsw_cti_nunique = unsw_train["cve_ctx_service"].nunique(dropna=True)
print(f"UNSW train: {len(unsw_train):,} рядків")
print(f"cve_ctx_service: {unsw_cti_na_rate:.2%} NaN, {unsw_cti_nunique} унікальних значень (non-NaN)")

unsw_results = {}
for native_col in NATIVE_COLS["UNSW"]:
    sep20 = "=" * 20
    print(f"\n{sep20} UNSW: {native_col} vs {CTI_COL} {sep20}")
    n_series, c_series = unsw_train[native_col], unsw_train[CTI_COL]

    h_native = entropy_bits(n_series)
    h_cti = entropy_bits(c_series)
    h_cti_given_native = conditional_entropy_bits(c_series, n_series)
    h_native_given_cti = conditional_entropy_bits(n_series, c_series)
    mi = mutual_information_bits(n_series, c_series)
    nmi = normalized_mutual_information(n_series, c_series)

    dom_native_to_cti = dominant_mapping(unsw_train, native_col, CTI_COL)
    dom_cti_to_native = dominant_mapping(unsw_train, CTI_COL, native_col)

    native_vals = set(n_series.dropna().unique())
    cti_vals = set(c_series.dropna().unique())
    literal_overlap_vals = native_vals & cti_vals
    literal_equal_share = float((n_series.astype("object") == c_series.astype("object")).mean())

    print(f"H({native_col})={h_native:.4f} bits   H({CTI_COL})={h_cti:.4f} bits")
    print(f"H({CTI_COL}|{native_col})={h_cti_given_native:.4f} bits  "
          f"(залишкова невизначеність CTI-service ПІСЛЯ того, як {native_col} вже відомий)")
    print(f"H({native_col}|{CTI_COL})={h_native_given_cti:.4f} bits")
    print(f"I({native_col};{CTI_COL})={mi:.4f} bits   NMI={nmi:.4f}")
    print(f"Літеральний overlap словників (case-sensitive): {len(literal_overlap_vals)} спільних значень "
          f"з {len(native_vals)} ({native_col}) / {len(cti_vals)} ({CTI_COL})")
    print(f"Частка рядків з ЛІТЕРАЛЬНО однаковим значенням {native_col}=={CTI_COL}: {literal_equal_share:.2%}")
    print(f"Найбільш домінантні {native_col}->{CTI_COL} (top 5 за group_total):")
    print(dom_native_to_cti.head(5).to_string(index=False))

    unsw_results[native_col] = dict(
        h_native=h_native, h_cti=h_cti, h_cti_given_native=h_cti_given_native,
        h_native_given_cti=h_native_given_cti, mutual_information=mi, normalized_mi=nmi,
        n_literal_overlap_values=len(literal_overlap_vals), literal_equal_share=literal_equal_share,
        dominant_native_to_cti=dom_native_to_cti, dominant_cti_to_native=dom_cti_to_native,
    )


UNSW train: 1,441,568 рядків
cve_ctx_service: 0.00% NaN, 28 унікальних значень (non-NaN)

==================== UNSW: state vs cve_ctx_service ====================
H(state)=1.0324 bits   H(cve_ctx_service)=2.1454 bits
H(cve_ctx_service|state)=1.5322 bits  (залишкова невизначеність CTI-service ПІСЛЯ того, як state вже відомий)
H(state|cve_ctx_service)=0.4193 bits
I(state;cve_ctx_service)=0.6131 bits   NMI=0.5939
Літеральний overlap словників (case-sensitive): 0 спільних значень з 16 (state) / 28 (cve_ctx_service)
Частка рядків з ЛІТЕРАЛЬНО однаковим значенням state==cve_ctx_service: 0.00%
Найбільш домінантні state->cve_ctx_service (top 5 за group_total):
state dominant    share  count  group_total  n_unique_to
  FIN    other 0.681110 688504      1010856           22
  CON      dns 0.664843 259064       389662           18
  INT      dns 0.497921  18326        36805            9
  REQ    other 0.991633   3437         3466            6
  RST     http 0.824742    240          291           

## 4. CIC: protocol vs cve_ctx_service (CIC не має native service-подібного поля)

In [5]:
cic_train = train_data["CIC"]
cic_cti_na_rate = cic_train["cve_ctx_service"].isna().mean()
cic_cti_nunique = cic_train["cve_ctx_service"].nunique(dropna=True)
print(f"CIC train: {len(cic_train):,} рядків")
print(f"cve_ctx_service: {cic_cti_na_rate:.2%} NaN, {cic_cti_nunique} унікальних значень (non-NaN)")

native_col = "protocol"
n_series, c_series = cic_train[native_col], cic_train[CTI_COL]

h_native = entropy_bits(n_series)
h_cti = entropy_bits(c_series)
h_cti_given_native = conditional_entropy_bits(c_series, n_series)
h_native_given_cti = conditional_entropy_bits(n_series, c_series)
mi = mutual_information_bits(n_series, c_series)
nmi = normalized_mutual_information(n_series, c_series)

dom_native_to_cti = dominant_mapping(cic_train, native_col, CTI_COL)
dom_cti_to_native = dominant_mapping(cic_train, CTI_COL, native_col)

native_vals = set(n_series.dropna().unique())
cti_vals = set(c_series.dropna().unique())
literal_overlap_vals = native_vals & cti_vals
literal_equal_share = float((n_series.astype("object") == c_series.astype("object")).mean())

print(f"H({native_col})={h_native:.4f} bits   H({CTI_COL})={h_cti:.4f} bits")
print(f"H({CTI_COL}|{native_col})={h_cti_given_native:.4f} bits  "
      f"(залишкова невизначеність CTI-service ПІСЛЯ того, як {native_col} вже відомий)")
print(f"H({native_col}|{CTI_COL})={h_native_given_cti:.4f} bits")
print(f"I({native_col};{CTI_COL})={mi:.4f} bits   NMI={nmi:.4f}")
print(f"Літеральний overlap словників (case-sensitive): {len(literal_overlap_vals)} спільних значень "
      f"з {len(native_vals)} ({native_col}) / {len(cti_vals)} ({CTI_COL})")
print(f"Частка рядків з ЛІТЕРАЛЬНО однаковим значенням {native_col}=={CTI_COL}: {literal_equal_share:.2%}")
print(f"Найбільш домінантні {native_col}->{CTI_COL} (top 5 за group_total):")
print(dom_native_to_cti.head(5).to_string(index=False))

cic_results = {native_col: dict(
    h_native=h_native, h_cti=h_cti, h_cti_given_native=h_cti_given_native,
    h_native_given_cti=h_native_given_cti, mutual_information=mi, normalized_mi=nmi,
    n_literal_overlap_values=len(literal_overlap_vals), literal_equal_share=literal_equal_share,
    dominant_native_to_cti=dom_native_to_cti, dominant_cti_to_native=dom_cti_to_native,
)}


CIC train: 1,979,346 рядків
cve_ctx_service: 0.00% NaN, 29 унікальних значень (non-NaN)
H(protocol)=0.9618 bits   H(cve_ctx_service)=2.1665 bits
H(cve_ctx_service|protocol)=1.2312 bits  (залишкова невизначеність CTI-service ПІСЛЯ того, як protocol вже відомий)
H(protocol|cve_ctx_service)=0.0265 bits
I(protocol;cve_ctx_service)=0.9352 bits   NMI=0.9724
Літеральний overlap словників (case-sensitive): 1 спільних значень з 3 (protocol) / 29 (cve_ctx_service)
Частка рядків з ЛІТЕРАЛЬНО однаковим значенням protocol==cve_ctx_service: 0.06%
Найбільш домінантні protocol->cve_ctx_service (top 5 за group_total):
protocol dominant    share  count  group_total  n_unique_to
     tcp     http 0.344432 425639      1235769           26
     udp      dns 0.959739 712415       742301            6
   other    other 1.000000   1276         1276            1


## 5. Підсумкова порівняльна таблиця UNSW vs CIC (для service/protocol -- прямий аналог)

Головне порівняння з дизайну: `service` (UNSW) vs `protocol` (CIC) -- найближчий структурний
аналог кожного датасету до CTI `cve_ctx_service`. `state`/`proto` (UNSW) виводяться окремо
вище як додатковий контекст, не як частина головного порівняння.

In [6]:
u = unsw_results["service"]
c = cic_results["protocol"]
summary_rows = [
    dict(metric="Native ознака (головне порівняння)", UNSW="service", CIC="protocol"),
    dict(metric="H(native) bits", UNSW=f"{u['h_native']:.4f}", CIC=f"{c['h_native']:.4f}"),
    dict(metric="H(cve_ctx_service) bits", UNSW=f"{u['h_cti']:.4f}", CIC=f"{c['h_cti']:.4f}"),
    dict(metric="H(cve_ctx_service | native) bits", UNSW=f"{u['h_cti_given_native']:.4f}",
         CIC=f"{c['h_cti_given_native']:.4f}"),
    dict(metric="I(native; cve_ctx_service) bits", UNSW=f"{u['mutual_information']:.4f}",
         CIC=f"{c['mutual_information']:.4f}"),
    dict(metric="NMI (0=незалежні, 1=повністю визначено)", UNSW=f"{u['normalized_mi']:.4f}",
         CIC=f"{c['normalized_mi']:.4f}"),
    dict(metric="Літеральна рівність native==cve_ctx_service", UNSW=f"{u['literal_equal_share']:.2%}",
         CIC=f"{c['literal_equal_share']:.2%}"),
]
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print("\nІнтерпретаційне правило (без апріорного висновку -- лише правило для читання чисел):")
print("  H(cve_ctx_service | native) близько до 0 і NMI близько до 1  => cve_ctx_service майже")
print("    повністю ВИЗНАЧАЄТЬСЯ native ознакою -- мало фізичного простору для нової інформації.")
print("  H(cve_ctx_service | native) близько до H(cve_ctx_service) і NMI близько до 0 => native")
print("    ознака майже нічого не каже про cve_ctx_service -- CTI приносить незалежну інформацію.")


                                     metric    UNSW      CIC
         Native ознака (головне порівняння) service protocol
                             H(native) bits  1.9093   0.9618
                    H(cve_ctx_service) bits  2.1454   2.1665
           H(cve_ctx_service | native) bits  0.6542   1.2312
            I(native; cve_ctx_service) bits  1.4911   0.9352
    NMI (0=незалежні, 1=повністю визначено)  0.7810   0.9724
Літеральна рівність native==cve_ctx_service  36.09%    0.06%

Інтерпретаційне правило (без апріорного висновку -- лише правило для читання чисел):
  H(cve_ctx_service | native) близько до 0 і NMI близько до 1  => cve_ctx_service майже
    повністю ВИЗНАЧАЄТЬСЯ native ознакою -- мало фізичного простору для нової інформації.
  H(cve_ctx_service | native) близько до H(cve_ctx_service) і NMI близько до 0 => native
    ознака майже нічого не каже про cve_ctx_service -- CTI приносить незалежну інформацію.


## 6. Збереження артефактів

In [7]:
for dataset_key, results in [("UNSW", unsw_results), ("CIC", cic_results)]:
    for native_col, r in results.items():
        r["dominant_native_to_cti"].to_csv(
            AUDIT_OUT_DIR / f"dominant_mapping_{dataset_key}_{native_col}_to_cti.csv", index=False)
        r["dominant_cti_to_native"].to_csv(
            AUDIT_OUT_DIR / f"dominant_mapping_{dataset_key}_cti_to_{native_col}.csv", index=False)

summary_df.to_csv(AUDIT_OUT_DIR / "overlap_audit_summary_table.csv", index=False)

overlap_summary_json = dict(
    UNSW={k: {kk: vv for kk, vv in v.items() if not isinstance(vv, pd.DataFrame)} for k, v in unsw_results.items()},
    CIC={k: {kk: vv for kk, vv in v.items() if not isinstance(vv, pd.DataFrame)} for k, v in cic_results.items()},
)
with open(AUDIT_OUT_DIR / "overlap_audit_summary.json", "w") as f:
    json.dump(overlap_summary_json, f, indent=2, default=str)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("*")):
    print(f"  {p.name}")


Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5e1_cti_native_overlap_audit:
  dominant_mapping_CIC_cti_to_protocol.csv
  dominant_mapping_CIC_protocol_to_cti.csv
  dominant_mapping_UNSW_cti_to_proto.csv
  dominant_mapping_UNSW_cti_to_service.csv
  dominant_mapping_UNSW_cti_to_state.csv
  dominant_mapping_UNSW_proto_to_cti.csv
  dominant_mapping_UNSW_service_to_cti.csv
  dominant_mapping_UNSW_state_to_cti.csv
  overlap_audit_summary.json
  overlap_audit_summary_table.csv


## Підсумок Phase 1

Заповнюється фактичним виводом після запуску на реальних даних: чи `H(cve_ctx_service |
service)` для UNSW близьке до нуля (гіпотеза overlap підтверджується -- native `service` вже
майже повністю визначає CTI-контекст) чи ні; те саме для CIC `protocol`; чи `NMI` для UNSW
суттєво вище, ніж для CIC (що дало б механістичне пояснення асиметрії ΔAP).

Це визначає, чи Phase 2 (feature attribution) варто інтерпретувати через призму overlap, і чи
Phase 3 (targeted ablation: `F0 + cve_ctx_service` окремо від `F0 + решта CTI`) взагалі
потрібен для розкладання внеску -- рішення приймається лише після цих чисел, не наперед.